In [ ]:
import pygame
import sys
import random

pygame.init()

# =========================
# CONFIGURAÇÕES DE TELA E CORES
# =========================
WIDTH, HEIGHT = 1280, 720
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("⚡ A Profecia dos Enigmas ⚡")
clock = pygame.time.Clock()

# Estados do Jogo
INPUT, MENU, GAME, RANK = "input", "menu", "game", "rank"
state = INPUT

# Variáveis do Jogo
player_name, coins, q_index, TOTAL_QUESTIONS = "", 0, 0, 10
game_started, current_q, answers, selected_answer, show_feedback, feedback_timer = False, None, [], None, False, 0

# Cores
BLUE_DARK, BLUE_LIGHT = (15, 32, 67), (25, 64, 131)
TEXT_WHITE, NEON_CYAN, NEON_PURPLE, HOVER_PURPLE = (240, 240, 245), (0, 245, 255), (130, 98, 221), (179, 145, 255)
GREEN_CORRECT, GREEN_BUTTON = (30, 130, 70), (56, 176, 0)
RED_WRONG, RED_BUTTON = (140, 35, 35), (255, 74, 74)
current_bg_color = BLUE_DARK

# Fontes
font_title = pygame.font.SysFont("impact", 60)
font_hud = pygame.font.SysFont("consolas", 24, bold=True)
font_buttons = pygame.font.SysFont("arial", 22, bold=True)

# =========================
# CLASSE: BOTÃO ESTILIZADO
# =========================
class Button:
    def __init__(self, x, y, w, h, text, color=NEON_PURPLE, text_color=TEXT_WHITE):
        self.rect = pygame.Rect(x, y, w, h)
        self.text = text
        self.color = color
        self.text_color = text_color

    def draw(self):
        mouse = pygame.mouse.get_pos()
        current_color = HOVER_PURPLE if self.rect.collidepoint(mouse) and self.color == NEON_PURPLE else self.color

        pygame.draw.rect(screen, current_color, self.rect, border_radius=12)
        pygame.draw.rect(screen, NEON_CYAN, self.rect, 2, border_radius=12)

        txt = font_buttons.render(self.text, True, self.text_color)
        screen.blit(txt, (self.rect.centerx - txt.get_width() // 2, self.rect.centery - txt.get_height() // 2))

    def click(self, event):
        return event.type == pygame.MOUSEBUTTONDOWN and event.button == 1 and self.rect.collidepoint(event.pos)

# Instanciação dos Botões fixos
btn_enter = Button(WIDTH//2 - 120, 440, 240, 60, "ENTRAR NO CLÃ")
btn_start = Button(WIDTH//2 - 160, 260, 320, 65, "🎮 INICIAR PROVA")
btn_resume = Button(WIDTH//2 - 160, 350, 320, 65, "▶️ RETOMAR")
btn_rank = Button(WIDTH//2 - 160, 440, 320, 65, "🏆 RANKING GLOBAL")
btn_exit = Button(WIDTH//2 - 160, 530, 320, 65, "❌ SAIR", color=(100, 30, 30))
btn_game_menu = Button(20, 20, 130, 40, "Menu")
btn_game_exit = Button(160, 20, 130, 40, "Sair", color=(100, 30, 30))

# Enigmas
enigma_pool = [
    ("O que nasce grande e morre pequeno?", "O lapis", "O ser humano", "A arvore"),
    ("Tenho chaves, mas nao abro portas. Tenho espaco, mas nao tenho salas?", "O teclado", "O hotel", "O chaveiro"),
    ("O que corre mas nao tem pernas, e tem leito mas nao dorme?", "O rio", "O vento", "O trem"),
    ("Faco duas pessoas a partir de uma. O que eu sou?", "O espelho", "A sombra", "A foto"),
    ("Se voce me soltar eu quebro, mas se sorrir para mim eu sorrio de volta.", "O espelho", "O vidro", "A agua"),
    ("O que esta sempre a caminho, mas nunca chega de fato?", "O amanha", "O carteiro", "O futuro distante"),
    ("Quanto mais voce tira de mim, maior eu fico. O que eu sou?", "Um buraco", "A areia", "A idade"),
    ("O que tem muitas palavras, mas nunca fala nada?", "O livro", "O dicionario", "A mente"),
    ("Estou sempre no passado, criado no presente, mudo o futuro.", "A escolha", "O vento", "O pensamento"),
    ("Tenho cidades, mas nao casas. Tenho montanhas, mas nao arvores.", "O mapa", "O espaco", "O deserto"),
    ("O que pode encher uma sala inteira sem ocupar nenhum espaco?", "A luz", "O som", "O vento"),
    ("Se voce me tem, quer compartilhar. Se compartilha, nao me tem mais.", "Um segredo", "O almoco", "Uma ideia")
]
active_questions = []
ranking = []

# =========================
# FUNÇÕES AUXILIARES / TELAS
# =========================
def draw_gradient_bg(color_top, color_bottom):
    for y in range(HEIGHT):
        alpha = y / HEIGHT
        r = int(color_top[0] * (1 - alpha) + color_bottom[0] * alpha)
        g = int(color_top[1] * (1 - alpha) + color_bottom[1] * alpha)
        b = int(color_top[2] * (1 - alpha) + color_bottom[2] * alpha)
        pygame.draw.line(screen, (r, g, b), (0, y), (WIDTH, y))

def load_question():
    global current_q, answers, selected_answer, show_feedback, active_questions, current_bg_color
    selected_answer, show_feedback, current_bg_color = None, False, BLUE_DARK

    if not active_questions:
        active_questions = list(enigma_pool)
        random.shuffle(active_questions)

    pergunta, correta, errada1, errada2 = active_questions.pop(0)
    opts = [correta, errada1, errada2]
    random.shuffle(opts)  
    
    current_q = (pergunta, opts, opts.index(correta))
    answers = [Button(WIDTH//2 - 250, 300 + i * 90, 500, 65, opt) for i, opt in enumerate(opts)]

def update_ranking(name, score):
    ranking.append((name, score))
    ranking.sort(key=lambda x: x[1], reverse=True)

def draw_input():
    draw_gradient_bg(BLUE_LIGHT, BLUE_DARK)
    t = font_title.render("QUEM MOSTRARA SUA MENTE?", True, NEON_CYAN)
    screen.blit(t, (WIDTH//2 - t.get_width()//2, 180))
    pygame.draw.rect(screen, (20, 40, 80), (WIDTH//2 - 200, 310, 400, 60), border_radius=10)
    pygame.draw.rect(screen, NEON_CYAN, (WIDTH//2 - 200, 310, 400, 60), 2, border_radius=10)
    n = font_hud.render(player_name + ("|" if pygame.time.get_ticks() % 1000 < 500 else ""), True, TEXT_WHITE)
    screen.blit(n, (WIDTH//2 - n.get_width()//2, 325))
    btn_enter.draw()

def draw_menu():
    draw_gradient_bg(BLUE_LIGHT, BLUE_DARK)
    t = font_title.render("A PROFECIA DOS ENIGMAS", True, NEON_CYAN)
    screen.blit(t, (WIDTH//2 - t.get_width()//2, 90))
    btn_start.draw()
    if game_started: btn_resume.draw()
    btn_rank.draw()
    btn_exit.draw()

def draw_game():
    screen.fill(current_bg_color) if show_feedback else draw_gradient_bg(BLUE_LIGHT, BLUE_DARK)
    hud = font_hud.render(f"NICK: {player_name}  |  🪙 COINS: {coins}  |  FASE: {q_index + 1}/{TOTAL_QUESTIONS}", True, NEON_CYAN)
    screen.blit(hud, (WIDTH//2 - hud.get_width()//2, 25))
    btn_game_menu.draw()
    btn_game_exit.draw()

    if current_q:
        pygame.draw.rect(screen, (10, 25, 60, 200), (WIDTH//2 - 500, 140, 1000, 110), border_radius=15)
        pygame.draw.rect(screen, NEON_CYAN, (WIDTH//2 - 500, 140, 1000, 110), 2, border_radius=15)
        q_surface = font_buttons.render(current_q[0], True, TEXT_WHITE)
        screen.blit(q_surface, (WIDTH//2 - q_surface.get_width()//2, 180))

        for i, b in enumerate(answers):
            b.color = GREEN_BUTTON if show_feedback and i == current_q[2] else (RED_BUTTON if show_feedback and i == selected_answer else NEON_PURPLE)
            b.draw()

def draw_rank():
    draw_gradient_bg(BLUE_LIGHT, BLUE_DARK)
    t = font_title.render("👑 LEADERBOARD", True, NEON_CYAN)
    screen.blit(t, (WIDTH//2 - t.get_width()//2, 80))
    y = 200
    if not ranking:
        empty = font_buttons.render("Nenhum recorde ainda. Seja o primeiro!", True, TEXT_WHITE)
        screen.blit(empty, (WIDTH//2 - empty.get_width()//2, y + 50))
    else:
        for i, (n, c) in enumerate(ranking[:7]):
            txt = font_hud.render(f"#{i+1}  {n.upper():<15} 🪙 {c} XP", True, TEXT_WHITE if i > 0 else NEON_CYAN)
            screen.blit(txt, (WIDTH//2 - 200, y))
            y += 55
    info = font_hud.render("[ Pressione ESC para Voltar ]", True, NEON_CYAN)
    screen.blit(info, (WIDTH//2 - info.get_width()//2, 620))

# =========================
# LOOP PRINCIPAL
# =========================
running = True
while running:
    clock.tick(60)
    current_time = pygame.time.get_ticks()

    if state == GAME and show_feedback and current_time > feedback_timer:
        q_index += 1
        if q_index >= TOTAL_QUESTIONS:
            update_ranking(player_name, coins)
            game_started, state = False, RANK
        else:
            load_question()

    for event in pygame.event.get():
        if event.type == pygame.QUIT: running = False

        if state == INPUT:
            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_BACKSPACE: player_name = player_name[:-1]
                elif event.key == pygame.K_RETURN and player_name.strip() != "": state = MENU
                elif len(player_name) < 12 and event.unicode.isalnum(): player_name += event.unicode
            if btn_enter.click(event) and player_name.strip() != "": state = MENU

        elif state == MENU:
            if btn_start.click(event):
                coins, q_index, active_questions, game_started, state = 0, 0, [], True, GAME
                load_question()
            if btn_resume.click(event) and game_started: state = GAME
            if btn_rank.click(event): state = RANK
            if btn_exit.click(event): running = False

        elif state == GAME:
            if btn_game_menu.click(event): state = MENU
            if btn_game_exit.click(event): running = False
            if not show_feedback:
                for i, b in enumerate(answers):
                    if b.click(event):
                        selected_answer, show_feedback, feedback_timer = i, True, current_time + 1200
                        if i == current_q[2]:
                            coins, current_bg_color = coins + 100, GREEN_CORRECT
                        else:
                            coins, current_bg_color = max(0, coins - 25), RED_WRONG

        elif state == RANK and event.type == pygame.KEYDOWN and event.key == pygame.K_ESCAPE:
            state = MENU

    if state == INPUT: draw_input()
    elif state == MENU: draw_menu()
    elif state == GAME: draw_game()
    elif state == RANK: draw_rank()
    pygame.display.flip()

pygame.quit()
sys.exit()